In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os

/home/aljebra/Generative AI tutorial/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
groq_api = os.getenv("GROQ_API_KEY")

In [ ]:
llm = ChatGroq(model = 'openai/gpt-oss-120b',

 api_key=groq_api,
 max_tokens=1024,
 temperature = 0.8 
 )

In [5]:
from langchain_core.messages import HumanMessage

In [6]:
response = llm.invoke([HumanMessage(content = " hello, My name is Ridwan, Nice to meet you!")])

In [7]:
from IPython.display import Markdown, display

In [8]:
display(Markdown(f"{response.content}"))

Hello Ridwan! Nice to meet you too. How can I assist you today?

In [9]:
from langchain_core.messages import AIMessage

In [10]:
# from langchain_core.messages import content


response = llm.invoke(
    [
        HumanMessage(content="hello ChatGpt, I'm Ridwan and I love building software!"),
        AIMessage(content = response.content),
        HumanMessage(content="What's my name and what do i do?")
    ]
)

In [11]:
display(Markdown(f"{response.content}"))

Your name is **Ridwan**, and you love **building software**—so you’re a software enthusiast/developer. Let me know if there’s anything specific you’d like to discuss or work on!

##### Let's implement message history component - which helps the model keep track of inputs and outputs and store them in some datastore. Future interraction will then load these massages and pass them into the chain as part of the input

In [12]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

### function to keep track of different chat session

In [13]:
#create dictionary to store messages
store_messages = {}

def get_session_history(session_id : str) -> BaseChatMessageHistory:
    #check if session id is not in store
    if session_id not in store_messages:
        #if not then pass chatmessagehistory object in the store using the session id as key
        store_messages[session_id] = ChatMessageHistory()

    #but if the session id is found in the store, it value (chatmessageHistory) should be returned
    return store_messages[session_id]


In [14]:
#Now get your history using Runnablewithmessagehistory
get_chat_history_with_runnable = RunnableWithMessageHistory(llm, get_session_history)

In [15]:
#configure a session id (hardcoded)
config = {"session_id": "chat1"}

In [16]:
#interracting with the model using the session id 
response = get_chat_history_with_runnable.invoke(
    [HumanMessage(content="Hello, my name is Ridwan and I love coding in Python")],
    config=config
)

In [17]:
display(Markdown(f"{response.content}"))

Hey Ridwan! 👋 Great to meet a fellow Python enthusiast. What kind of projects do you enjoy working on? Are you into web development with Flask/Django, data science with pandas and NumPy, automation scripts, or something else? I'd love to hear about your favorite Python experiences!

In [18]:
#check if it will remember what my name is using the same config
#interracting with the model using the session id 
response = get_chat_history_with_runnable.invoke(
    [HumanMessage(content="what is my name?")],
    config=config
)

In [19]:
display(Markdown(f"{response.content}"))

Your name is Ridwan.

### Now let's try and change the session id in the config to check if it will remember my name again

In [20]:
#configure a new session id (hardcoded)
config =  {"session_id": "chat2"}

#check if it will remember what my name is using another config
#interracting with the model using the session id 
response = get_chat_history_with_runnable.invoke(
    [HumanMessage(content="what is my name?")],
    config=config
)

In [21]:
display(Markdown(f"{response.content}"))

I don’t have any information about your name. If you’d like me to address you a certain way, just let me know!

### Using prompt template 

In [22]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [23]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", " As an AI assistant, answer the given questions to the best of your ability"),
        #any human message we give must be in key-value pair with the keyword "messages" which is different from what we did above
        MessagesPlaceholder(variable_name='messages'),
    ]
)

In [ ]:
#now let's chain
chain = prompt_template | llm

In [25]:
response = chain.invoke({"messages": [HumanMessage(content="Hi, My name is Ridwan")]})

In [26]:
display(Markdown(f"{response.content}"))

Hello Ridwan! Nice to meet you. How can I assist you today?

In [27]:
#to keep track of session invoke using Runnablewithmessagehistory
runnable_with_message_history = RunnableWithMessageHistory(chain, get_session_history=get_session_history)

In [28]:
#configure another session id (hardcode)
config =  {"session_id": "chat3"}

In [29]:
#check if it will remember what my name is using the another config
#interracting with the model using the session id 
response = runnable_with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Ridwan")],
    config=config
)

### Now let's have more input variable to the prompt beyond it's default variable (this will make it two)

In [30]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", " As an AI assistant, answer the given questions to the best of your ability in this {language}"),
        #any human message we give must be in key-value pair with the keyword "messages" which is different from what we did above
        MessagesPlaceholder(variable_name='messages'),
        
    ]
)

In [31]:
chain = prompt_template | llm

In [32]:
#Now update the prompt_template to take another input, in this case - language 

response = chain.invoke({"messages": [HumanMessage(content="Hi, My name is Ridwan")], "language": "French"})

display(Markdown(f"{response.content}"))

Bonjour Ridwan ! Enchanté de faire votre connaissance. Comment puis-je vous aider aujourd’hui ?

##### Let's now wrap this more complicated chain in a message history class. This time because there are multiple keys in the input, we need to specify. We need to specify the correct to save the chat history 

In [33]:
#now use the input_messages_key with the correct input that will be used to save the message history since we now have two input variables in our template
runnable_with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages" #using input message key
)

In [34]:
#configure another session id (hardcode)
config =  {"session_id": "chat4"}

In [35]:
#check if it will remember what my name is using the another config
#interracting with the model using the session id 
response = runnable_with_message_history.invoke(
    {"messages":[HumanMessage(content="Hi, My name is Ridwan")], "language" : "french"},
    config=config
)

display(Markdown(f"{response.content}"))


Bonjour Ridwan ! Enchanté de faire votre connaissance. Comment puis-je vous aider aujourd'hui ?

In [36]:
response = runnable_with_message_history.invoke(
    {"messages":[HumanMessage(content="what's my name?")], "language" : "yoruba"},
    config=config
)

display(Markdown(f"{response.content}"))


Your name is Ridwan.

#### Managing the conversation history - this is important to prevent the overflow of messages beyond the LLM context window using "trim_messages" package

In [37]:
from langchain_core.messages import SystemMessage, trim_messages

In [38]:
trimmer = trim_messages(
    max_tokens = 70, # to trim you can reduce the token from 70 to 45
    strategy = 'last',
    token_counter = llm,
    include_system = True,
    allow_partial = False,
    start_on = 'human'
)

In [39]:
messages = [
    SystemMessage(content = "You're an AI assistant"),
    HumanMessage(content= "Hi, I'm Ridwan"),
    AIMessage(content="Hi!"),
    HumanMessage(content= "I like solving math problem"),
    AIMessage(content="That's impressive as math can help improve your thinking ability"),
    HumanMessage(content= "what is 2 + 2"),
    AIMessage(content='4'),
    HumanMessage(content='Thanks for this wonderful conversation'),
    AIMessage(content="You're welcome, I'm happy to help everytime!")
]

In [40]:
trimmer.invoke(messages) #This will reduce the output compare to 70 max_token we used earlier

[SystemMessage(content="You're an AI assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="Hi, I'm Ridwan", additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hi!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like solving math problem', additional_kwargs={}, response_metadata={}),
 AIMessage(content="That's impressive as math can help improve your thinking ability", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Thanks for this wonderful conversation', additional_kwargs={}, response_metadata={}),
 AIMessage(content="You're welcome, I'm happy to help everytime!", additional_kwargs={}, response_metadata={})]

### you can also use chain to pass this trimmer function

In [41]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

In [43]:
chain = (
    RunnablePassthrough.assign(messages = itemgetter("messages")|trimmer) | prompt_template | llm
)

In [44]:
response = chain.invoke(
    {
    "messages": messages + [HumanMessage(content="what is 2 + 2")],
    "language" : "French"
    }
)

In [45]:
display(Markdown(f"{response.content}"))

2 + 2 = 4.

### Now let's wrap this in a message history

In [46]:
runnable_with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages" #using input message key
)

In [47]:
#configure another session id (hardcode)
config = {"session_id": "chat5"}

In [48]:
response = runnable_with_message_history.invoke(
    {"messages":[HumanMessage(content="what's my name?")], "language" : "yoruba"},
    config=config
)

display(Markdown(f"{response.content}"))


Emi kò mọ orúkọ rẹ. Ṣé o lè sọ fún mi?